# Optimisation du preproc

In [48]:
import pandas as pd
data = pd.read_csv("../../raw_data/recipes_ingredients.csv")

In [49]:
data.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


In [50]:
data[["persons", "portion_size"]] = data["serving_size"].str.extract(
    r"(\d+)\s*\(([^)]+)\)")
data = data.drop(columns="serving_size", axis=1)

In [51]:
data = data[data["servings"] <= 50]

In [52]:
import ast
def safe_literal_eval(value):
    """
    Transforme les strings en liste de string
    """

    try:
        result = ast.literal_eval(value)

        if isinstance(result, list):
            return result

        return []

    except (ValueError, SyntaxError, TypeError):
        return []

In [53]:
def transform_str_to_list(data):
    data["ingredients"] = data["ingredients"].apply(safe_literal_eval)
    data["ingredients_raw"] = data["ingredients_raw"].apply(safe_literal_eval)
    data["steps"] = data["steps"].apply(safe_literal_eval)
    data["tags"] = data["tags"].apply(safe_literal_eval)
    #Vire les listes vides
    data = data[data["ingredients"].apply(len) > 2]
    data = data[data["ingredients"].apply(len) < 15]
    data = data[data["ingredients_raw"].apply(len) > 2]
    data = data[data["ingredients_raw"].apply(len) < 15]
    data = data[data["steps"].apply(len) > 1]
    data = data[data["tags"].apply(len) > 0]
    return data

In [54]:
data = transform_str_to_list(data)

In [55]:
type(data.iloc[0].ingredients)

list

In [56]:
from mastershelf.recipes.params import *
import re


def clean_ingredient(text):
    text = text.lower().strip()


    text = re.sub(r"\b\d+([./]\d+)?\b", " ", text)

    for word in WORDS_TO_REMOVE:
        text = re.sub(rf"\b{re.escape(word)}\b", " ", text)


    text = re.sub(r"[,()]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [57]:
data["ingredients_clean"] = data["ingredients"].apply(
    lambda ingredients: [
        clean_ingredient(x)
        for x in ingredients
    ]
)

In [59]:
data.shape

(336432, 11)

In [60]:
unique_clean = (
    data["ingredients_clean"]
    .explode()
    .dropna()
    .unique()
)

len(unique_clean)

145425

In [78]:
ingredient_counts = (
    data["ingredients_clean"]
    .explode()
    .value_counts()
)
most_commun = ingredient_counts.head(4800).index.tolist()


In [79]:
with open('../../raw_data/your_file.txt', 'w') as f:
    for line in most_commun:
        f.write(f"{line}\n")

In [62]:
import json

with open("../../raw_data/agg_ing_mapping.json", "r", encoding="utf-8") as f:
    ingredient_mapping = json.load(f)

In [63]:
data["ingredients_clean"] = data["ingredients_clean"].apply(
    lambda ingredients: [
        ingredient_mapping.get(ingredient, ingredient)
        for ingredient in ingredients
    ]
)

In [67]:
ingredient_counts = (
    data["ingredients_clean"]
    .explode()
    .value_counts()
)
most_commun = ingredient_counts.head(3360).index.tolist()

In [69]:
with open('../../raw_data/most_communV2.txt', 'w') as f:
    for line in most_commun:
        f.write(f"{line}\n")

In [65]:
ingredient_counts = (
    data["ingredients_clean"]
    .explode()
    .value_counts()
)

In [66]:
cumulative = ingredient_counts.cumsum() / ingredient_counts.sum()

print("80% coverage:", (cumulative <= 0.80).sum())
print("90% coverage:", (cumulative <= 0.90).sum())
print("95% coverage:", (cumulative <= 0.95).sum())

80% coverage: 391
90% coverage: 3360
95% coverage: 28202


In [70]:
most_commons_set = set(most_commun)

df_filtered = data[
    data["ingredients"].apply(
        lambda ingredients: set(ingredients).issubset(most_commons_set)
    )
].copy()

In [ ]:
print(len(data))
print(len(df_filtered))

print(
    f"{len(df_filtered) / len(data) * 100:.2f}% des recettes conservées"
)

336432
14193
4.22% des recettes conservées


In [80]:
df_filtered.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size,ingredients_clean
63,420900,Delicious Orange Chocolate Muffins,This is an adaption of a friend's. My nephew d...,"[flour, baking soda, baking powder, chocolate ...","[1 1/2 cups sifted flour, 1 teaspoon ...","[Set 12 cup cake liners in a muffin pan., Preh...",12.0,"[60-minutes-or-less, time-to-make, course, cui...",1,79 g,"[flour, baking soda, baking powder, chocolate ..."
90,235622,Sweet Dessert Panini,Just saw this on Paula's Home cooking with Pau...,"[chocolate hazelnut, white bread, bananas, mar...","[1 cup chocolate hazelnut spread, 8 sli...","[Preheat the grill to medium-low., Heat hazeln...",1.0,"[15-minutes-or-less, time-to-make, preparation...",1,844 g,"[chocolate hazelnut, white bread, banana, mars..."
91,435924,Frozen Fruit Bites,From Family Fun Magazine - Cooking time is fre...,"[vanilla wafer cookies, vanilla yogurt, cream ...","[12 vanilla wafer cookies, 1/2 cup v...","[Place liners in a mini-cupcake pan., Put a wa...",12.0,"[time-to-make, course, main-ingredient, prepar...",1,38 g,"[vanilla wafer cookies, vanilla yogurt, cream ..."
161,219381,Baked Snapper With Chipotle Butter,Cooking Light,"[cumin, paprika, black pepper, red snapper fil...","[1/2 teaspoon ground cumin, 1/2 teaspoon ...","[Preheat oven to 400°., Combine the first 4 in...",4.0,"[30-minutes-or-less, time-to-make, main-ingred...",1,174 g,"[cumin, paprika, black pepper, red snapper fil..."
267,453368,Blueberry Muffins,Spring is here! Woo Hoo! fresh blueberries.,"[flour, baking powder, buttermilk, blueberries]","[2/3 cup shortening, 1 cup sugar, 3 ...",[cream the shortening and sugar till fluffy. a...,1.0,"[60-minutes-or-less, time-to-make, course, pre...",1,1416 g,"[flour, baking powder, buttermilk, blueberry]"
